# eosframes — Demo Notebook

This notebook showcases the main features of `eosframes`, a library for manipulating inputs and outputs from the [Ersilia Model Hub](https://github.com/ersilia-os/ersilia).

We use two real model output files bundled in this repository:

| File | Model | Description | Rows | Features |
|------|-------|-------------|------|----------|
| `data/example_eos4e40_v1.csv` | [eos4e40](https://github.com/ersilia-os/eos4e40) | Inhibition at 50 µM | 100 | 1 |
| `data/example_eos7m30_v1.csv` | [eos7m30](https://github.com/ersilia-os/eos7m30) | 49 ADMET properties (TDC benchmark) | 100 | 49 |

In [ ]:
import os
import json
import shutil
import tempfile

import numpy as np
import pandas as pd

import eosframes
from eosframes import (
    read_csv, read_h5, write_csv, write_h5,
    hstack, vstack,
    split_csv, convert_file, stack_files, append_files, dedupe_file,
    fit_scaler, apply_scaler, fit_scaler_file, apply_scaler_file,
    parse_name, make_output_name, is_valid_name,
)

# Paths to reference data
DATA_DIR = os.path.join("..", "data")
EOS4E40_CSV = os.path.join(DATA_DIR, "example_eos4e40_v1.csv")
EOS7M30_CSV = os.path.join(DATA_DIR, "example_eos7m30_v1.csv")

# Working directory for demo outputs
WORK_DIR = tempfile.mkdtemp(prefix="eosframes_demo_")
print("Working directory:", WORK_DIR)

---
## 1. Naming Convention

eosframes enforces a naming convention on all output files: `[prefix_]<model_id>_<version>.<ext>`.

Prefixes (like dates or project names) are allowed before the model identifier.

In [ ]:
# Parse any valid name
examples = [
    "eos4e40_v1.csv",
    "eos7m30_v2.h5",
    "eos4e40_v1_chunks",
    "260313_gardp_eos4e40_v1.csv",    # date + project prefix
    "example_eos7m30_v1.csv",          # descriptive prefix
    "output.csv",                       # invalid — no model_id
]

for name in examples:
    result = parse_name(name)
    print(f"{name!s:<40} → {result}")

In [ ]:
# Validate names and build canonical filenames
print("is_valid_name('example_eos4e40_v1.csv'):", is_valid_name("example_eos4e40_v1.csv"))
print("is_valid_name('output.csv'):             ", is_valid_name("output.csv"))
print()
print("make_output_name('eos4e40', 'v1', 'csv'):", make_output_name("eos4e40", "v1", "csv"))
print("make_output_name('eos7m30', 'v1', 'h5'): ", make_output_name("eos7m30", "v1", "h5"))

---
## 2. Reading and Writing Files

In [ ]:
# Read the eos4e40 output (1 feature: inhibition at 50 µM)
df4e40 = read_csv(EOS4E40_CSV)
print(f"Model: {df4e40.model_id}")
print(f"Shape: {df4e40.shape}")
df4e40.head()

In [ ]:
# Read the eos7m30 output (49 ADMET properties)
df7m30 = read_csv(EOS7M30_CSV)
print(f"Model: {df7m30.model_id}")
print(f"Shape: {df7m30.shape}")
feat_cols = [c for c in df7m30.columns if c not in {"key", "input"}]
print(f"Features ({len(feat_cols)}): {feat_cols[:5]} ...")
df7m30[["key", "input", "molecular_weight", "logp", "qed"]].head()

In [ ]:
# Write to H5 and read back
h5_path = os.path.join(WORK_DIR, "eos4e40_v1.h5")
write_h5(df4e40, h5_path, dtype=np.float32)
df_back = read_h5(h5_path)
print(f"Round-trip OK: {df_back.shape}, model_id={df_back.model_id}")
print(f"Max abs diff (float32 precision): {abs(df_back['inhibition_50um'].values - df4e40['inhibition_50um'].values).max():.6f}")

---
## 3. Splitting into Chunks

In [ ]:
# Split eos4e40 output (100 rows) into chunks of 25
chunks_dir = os.path.join(WORK_DIR, "eos4e40_chunks")
n = split_csv(EOS4E40_CSV, chunks_dir, chunksize=25)
print(f"Created {n} chunk files:")
for f in sorted(os.listdir(chunks_dir)):
    df_chunk = pd.read_csv(os.path.join(chunks_dir, f))
    print(f"  {f}: {len(df_chunk)} rows")

---
## 4. Converting Between Formats

In [ ]:
# CSV → H5
h5_out = os.path.join(WORK_DIR, "eos7m30_v1.h5")
convert_file(EOS7M30_CSV, h5_out)
print(f"CSV → H5: {os.path.getsize(h5_out):,} bytes")

# H5 → CSV
csv_out = os.path.join(WORK_DIR, "eos7m30_v1_restored.csv")
convert_file(h5_out, csv_out)
df_restored = pd.read_csv(csv_out)
print(f"H5 → CSV: {len(df_restored)} rows, {len(df_restored.columns)} columns")

# Chunks → H5
h5_from_chunks = os.path.join(WORK_DIR, "eos4e40_from_chunks_v1.h5")
convert_file(chunks_dir, h5_from_chunks)
df_assembled = read_h5(h5_from_chunks)
print(f"Chunks → H5: {df_assembled.shape}")

---
## 5. Horizontal Stacking (multiple models, same molecules)

In [ ]:
# Write copies to WORK_DIR (write_csv enforces naming convention)
p4 = os.path.join(WORK_DIR, "eos4e40_v1.csv")
p7 = os.path.join(WORK_DIR, "eos7m30_v1.csv")
write_csv(df4e40, p4)
write_csv(df7m30, p7)

# Stack the two models side by side
stacked_path = os.path.join(WORK_DIR, "stacked.csv")
stack_files([p4, p7], stacked_path, suffix=True)

stacked = pd.read_csv(stacked_path)
print(f"Stacked shape: {stacked.shape}")
print(f"Columns: {list(stacked.columns[:5])} ... (total {len(stacked.columns)})")
stacked[["key", "inhibition_50um.eos4e40", "molecular_weight.eos7m30", "logp.eos7m30", "qed.eos7m30"]].head()

In [ ]:
# DataFrame API: hstack
combined = hstack([df4e40, df7m30])
print(f"hstack result: {combined.shape}")
print("Feature columns from eos4e40:", [c for c in combined.columns if "eos4e40" in c])
print("First 3 feature columns from eos7m30:", [c for c in combined.columns if "eos7m30" in c][:3])

---
## 6. Vertical Appending (same model, multiple batches)

In [ ]:
# Split eos4e40 into two halves and append them back
half = len(df4e40) // 2
p_b1 = os.path.join(WORK_DIR, "eos4e40_v1_b1.csv")
p_b2 = os.path.join(WORK_DIR, "eos4e40_v1_b2.csv")
df4e40.iloc[:half].to_csv(p_b1, index=False)
df4e40.iloc[half:].to_csv(p_b2, index=False)

appended_path = os.path.join(WORK_DIR, "eos4e40_v1_appended.csv")
append_files([p_b1, p_b2], appended_path)

df_appended = pd.read_csv(appended_path)
print(f"Original rows: {len(df4e40)}, Appended rows: {len(df_appended)}")
assert list(df_appended["key"]) == list(df4e40["key"]), "Row order not preserved!"
print("Row order preserved: OK")

---
## 7. Deduplication

In [ ]:
# Introduce some duplicate rows
df_dup = pd.concat([df4e40, df4e40.iloc[:10]], ignore_index=True)
raw_path = os.path.join(WORK_DIR, "eos4e40_v1_raw.csv")
df_dup.to_csv(raw_path, index=False)
print(f"With duplicates: {len(df_dup)} rows")

deduped_path = os.path.join(WORK_DIR, "eos4e40_v1_deduped.csv")
before, after = dedupe_file(raw_path, deduped_path)
print(f"After dedupe: {after} rows (removed {before - after} duplicates)")

---
## 8. Standard Scaler

Fit a standard scaler on the eos7m30 ADMET outputs (49 features) and apply it to new data.

In [ ]:
# Fit scaler on the DataFrame directly
params = fit_scaler(df7m30)
print(f"Method: {params['method']}")
print(f"Fitted columns: {len(params['columns'])}")
print(f"Skipped columns (>25% missing): {params['skipped_columns']}")
print()

# Show parameters for a few columns
for col in ["molecular_weight", "logp", "qed"]:
    p = params["parameters"][col]
    print(f"  {col:<30}  mean={p['mean']:.3f}  std={p['std']:.3f}")

In [ ]:
# Apply scaler — verify zero mean and unit std
scaled = apply_scaler(df7m30, params)

print("After scaling (should be ~0 mean, ~1 std for each feature):")
for col in ["molecular_weight", "logp", "qed"]:
    print(f"  {col:<30}  mean={scaled[col].mean():.6f}  std={scaled[col].std(ddof=0):.6f}")

In [ ]:
# File API: fit and save transformer JSON
json_path = os.path.join(WORK_DIR, "eos7m30_v1_scaler.json")
scaled_path = os.path.join(WORK_DIR, "eos7m30_v1_scaled.csv")
out = fit_scaler_file(p7, json_path, output_path=scaled_path)
print(f"Scaled file written to: {out}")

# Inspect the transformer JSON
with open(json_path) as f:
    t = json.load(f)
print(f"\nTransformer JSON:")
print(f"  model_id:  {t['model_id']}")
print(f"  version:   {t['version']}")
print(f"  n_rows:    {t['n_rows']}")
print(f"  fitted_at: {t['fitted_at']}")
print(f"  columns:   {t['columns'][:3]} ... ({len(t['columns'])} total)")

In [ ]:
# Apply the saved transformer to new data
applied_path = os.path.join(WORK_DIR, "eos7m30_v1_applied.csv")
apply_scaler_file(p7, json_path, applied_path)

df_applied = pd.read_csv(applied_path)
df_scaled  = pd.read_csv(scaled_path)
print("Fit output == Apply output:", np.allclose(df_applied["molecular_weight"].values, df_scaled["molecular_weight"].values))

---
## 9. Hub Data

Fetch model metadata and column definitions from GitHub (requires network access).

In [ ]:
from eosframes import fetch_metadata, fetch_columns

# Fetch metadata for eos4e40
try:
    metadata = fetch_metadata("eos4e40")
    for key in ["Identifier", "Slug", "Title", "Task", "Input", "Output Type"]:
        if key in metadata:
            print(f"{key}: {metadata[key]}")
except Exception as e:
    print(f"(skipped — network unavailable: {e})")

In [ ]:
# Fetch run_columns.csv for eos7m30 v1
try:
    columns_df = fetch_columns("eos7m30", "v1")
    print(f"run_columns.csv: {len(columns_df)} rows")
    print(columns_df.head())
except Exception as e:
    print(f"(skipped — network unavailable: {e})")

---
## 10. Logging

All eosframes operations emit structured log messages via the Python `logging` module.

In [ ]:
import logging
from eosframes import get_logger

logger = get_logger()
logger.setLevel(logging.DEBUG)   # increase verbosity

# Now all operations will log at DEBUG level
_ = read_csv(EOS4E40_CSV)

logger.setLevel(logging.INFO)    # reset to default

---
## Clean up

In [ ]:
shutil.rmtree(WORK_DIR)
print("Temporary files removed.")